# System Prompt Engineering — Practical Notebook
## STUDENT VERSION — fill in every `TODO`

**Companion lab: shaping model behavior through the system prompt, not just the user message.**

---

A **system prompt** sets the model's standing behavior *before* the conversation starts — its role, its rules, even how it should reason — separately from whatever the user asks. The same user message can produce very different answers depending on what's in the system prompt. This lab builds that skill across three techniques:

- **Exercise 1 — System Prompt Engineering.** Going from no system prompt to one that encodes role, context, constraints, and output format.
- **Exercise 2 — Few-Shot Prompting via the system prompt.** Teaching a classification task by putting labelled examples *in the system prompt*, so any new ticket can be sent as a plain user message.
- **Exercise 3 — Chain-of-Thought via the system prompt.** Making step-by-step reasoning a *standing instruction* in the system prompt rather than something you repeat per question.

### How to use this notebook
Cells marked **`# TODO`** are yours to complete. The TODOs are about **writing system prompts and comparing outputs** — the plumbing (the API call helper, printing, etc.) is given so you can focus on the system prompt text itself, which is the actual skill being practiced.


## 1 · Setup & install

We use **LangChain** as a thin, standard interface over the LLM, and **Google's Gemini** as the model — Gemini has a free API tier, so no billing setup is needed to run this lab.

We install two packages: `langchain-google-genai` (LangChain's Gemini integration) and `langchain-core` (pulled in automatically as a dependency). If you are running locally, uncomment the `pip install` line. On Colab, run it as-is the first time.

You'll also need a free API key from **[Google AI Studio](https://aistudio.google.com/apikey)** (sign in, click "Create API key"). Paste it when prompted below — it will not be echoed to the notebook, and it is not saved anywhere on disk.


In [ ]:
# !pip install -q langchain-google-genai langchain-core

import getpass
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import SystemMessage, HumanMessage

if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Paste your Google AI Studio API key: ")

MODEL = "gemini-3.5-flash-lite"  # fast, cheap, and on Google's free tier

llm = ChatGoogleGenerativeAI(model=MODEL, temperature=0.7)

print("LangChain + Gemini client ready. Model:", MODEL)


## 2 · A tiny helper: `call_llm`

Every exercise below sends a **system prompt** + a **user prompt** and prints the reply. Rather than repeating the LangChain boilerplate in every cell, we wrap it once. This mirrors how you'd use LangChain in a real project — one thin wrapper function, called everywhere.

`call_llm` takes a **user prompt** and a **system prompt** (this time not optional — it's the main thing every exercise manipulates), builds a LangChain message list, and returns the model's text reply.

One Gemini-specific detail: recent Gemini models return `response.content` as a **list** of content blocks (not a plain string), so we pull the text out of the first block with `response.content[0]["text"]`.


In [ ]:
def call_llm(user_prompt, system_prompt):
    """Send a system prompt + user prompt to the model through LangChain,
    return the reply text."""
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=user_prompt),
    ]

    response = llm.invoke(messages)
    return response.content[0]["text"]


# Quick smoke test -- if this prints a normal sentence, you're wired up correctly.
print(call_llm("Say hello in exactly five words.", system_prompt="You are a friendly assistant."))


## 3 · Exercise 1 — System Prompt Engineering

> *Core idea:* the same user question can be answered vaguely or precisely, purely based on what's sitting in the **system prompt**. Four ingredients turn a vague system prompt into a precise one:
> - **Role** — who the model should act as ("You are an experienced Python instructor...").
> - **Context** — relevant details about the situation (who the plan is for, their background).
> - **Constraints** — hard requirements (time budget, tools allowed, length).
> - **Output format** — exactly how the answer should be structured (a table, numbered weeks, JSON, etc.).
>
> A system prompt with none of these forces the model to *guess* what you want, using only the bare user question. A system prompt with all four leaves little room for the model to guess wrong — and it does this *without changing the user's question at all*.

**Task:** ask an LLM for a 4-week Python study plan for a beginner — first with an empty/minimal system prompt, then with a system prompt carrying role, context, constraints, and output format.


### TODO 3.1 — Minimal system prompt

Keep the user question fixed: `"Create a 4-week Python study plan for a beginner."` For this first call, use the most generic system prompt possible (e.g. `"You are a helpful assistant."`) so nothing about role, context, constraints, or format is specified. This is your baseline to compare against later.


In [ ]:
user_question = "Create a 4-week Python study plan for a beginner."

# TODO 3.1 -- Write a minimal, generic system prompt (one short sentence,
# no role/context/constraints/format specifics -- e.g. "You are a helpful
# assistant.") Store it in `basic_system_prompt`, call it, and print the reply.

basic_system_prompt = ...  # TODO: your minimal system prompt string

basic_reply = call_llm(user_question, system_prompt=basic_system_prompt)
print(basic_reply)


Look at what came back. It's probably *fine* — but notice what's missing or inconsistent: Does it assume a background you didn't specify? Is the depth right for a total beginner? Is the plan easy to scan, or a wall of prose? These gaps are exactly what the four ingredients fix.

### TODO 3.2 — Engineered system prompt

Keep `user_question` **exactly the same** — you are only changing the system prompt. Write a new system prompt that deliberately includes all four ingredients:
- **Role**: give the model a persona (e.g. an experienced instructor who teaches absolute beginners).
- **Context**: describe the learner (e.g. no coding background, ~5 hours/week available).
- **Constraints**: things the plan must respect (must fit in 4 weeks, no paid resources, beginner-only topics).
- **Output format**: specify the structure you want back (e.g. a markdown table with columns Week / Topics / Practice Exercise, or numbered week headings).


In [ ]:
# TODO 3.2 -- Rewrite the SYSTEM PROMPT with role, context, constraints, and
# output format. The user question stays exactly `user_question` from above --
# only the system prompt changes.
#
# Suggested structure (feel free to diverge):
#   role:        "You are ..."
#   context:     "The learner is ..."
#   constraints: "Any plan you produce must ..."
#   format:      "Always return plans as ..."

engineered_system_prompt = ...  # TODO: your full system prompt string

engineered_reply = call_llm(user_question, system_prompt=engineered_system_prompt)
print(engineered_reply)


### TODO 3.3 — Compare

In a comment, note **two concrete differences** between `basic_reply` and `engineered_reply` (e.g. structure, level of detail, whether it respects a constraint you gave) — remembering that the *user question was identical* both times. This comparison is the actual point of the exercise: the system prompt alone is powerful enough to reshape the answer.


In [ ]:
# TODO 3.3 -- In a comment (or a couple of print statements), note two
# concrete differences between basic_reply and engineered_reply.
#
# e.g.:
# 1) ...
# 2) ...


## 4 · Exercise 2 — Few-Shot Prompting via the System Prompt

> *Core idea:* instead of *describing* a task in words, you can *demonstrate* it with a handful of labelled examples and let the model infer the pattern. Putting those examples **in the system prompt** (rather than repeating them in every user message) means the model behaves consistently for *any* new ticket a user sends — the examples become a standing part of its behavior, not a one-off instruction.
>
> A few-shot **system** prompt is: a short instruction, then N example inputs each followed by their correct label. The **user prompt** then just contains the new, unlabeled ticket — nothing else.

**Task:** classify support tickets into `low`, `medium`, or `high` urgency using a few-shot system prompt.


### TODO 4.1 — Pick and label three example tickets

Write three short, realistic support tickets and assign each one a label: `low`, `medium`, or `high`. Try to pick examples that are genuinely different in urgency (e.g. a cosmetic UI question vs. a billing question vs. a total outage) so the pattern you're teaching is clear.


In [ ]:
# TODO 4.1 -- Write three example tickets and label each with its urgency.
#
# Each entry is a dict: {"ticket": "...", "label": "low" | "medium" | "high"}
# Pick tickets that are clearly different in urgency from each other.

few_shot_examples = [
    {"ticket": ..., "label": ...},  # TODO
    {"ticket": ..., "label": ...},  # TODO
    {"ticket": ..., "label": ...},  # TODO
]

for ex in few_shot_examples:
    print(f"[{ex['label']:>6}] {ex['ticket']}")


### TODO 4.2 — Build the few-shot system prompt and test it

Assemble `few_shot_examples` into a **system prompt**: a short instruction line, then each example formatted consistently (e.g. `Ticket: ...` / `Urgency: ...`). Loop over `few_shot_examples` to build this block programmatically — don't hand-type each one into the string, or the prompt won't scale if you add a fourth example later.

The **user prompt** is then just the new ticket, with no extra formatting — the system prompt alone should be enough for the model to know what to do with it.


In [ ]:
new_ticket = "My dashboard shows last month's numbers with slightly outdated colors -- looks a bit off but nothing is broken."

# TODO 4.2 -- Build the few-shot SYSTEM prompt string.
#
# 1. Start with a short instruction, e.g.:
#    "Classify each support ticket's urgency as low, medium, or high. "
#    "Respond with a single word: low, medium, or high.\n\n"
# 2. Loop over `few_shot_examples` and append each one in a CONSISTENT
#    format, e.g.:
#        Ticket: <ticket text>
#        Urgency: <label>
#
# The USER prompt is just `new_ticket` -- no extra formatting needed, since
# the pattern already lives in the system prompt.

few_shot_system_prompt = ...  # TODO: build the full string, e.g. via a loop + f-strings

print(call_llm(new_ticket, system_prompt=few_shot_system_prompt))


### TODO 4.3 — Sanity check

Does the label the model gave `new_ticket` match what *you* would have assigned by hand? If not, look back at your three examples — is the boundary between `low` and `medium` actually clear from them, or could a reasonable person read it either way? Note your answer in a comment.


In [ ]:
# TODO 4.3 -- In a comment, say whether the model's label matches your own
# judgement, and if not, what about your three examples might have made
# the boundary ambiguous.


## 5 · Exercise 3 — Chain-of-Thought via the System Prompt

> *Core idea:* for problems that need multiple steps (arithmetic, logic, multi-part word problems), asking directly for the final answer forces the model to produce it in one shot, with no intermediate work — that's when careless mistakes creep in. Making **"reason step by step before answering"** a *standing rule in the system prompt* means every question you send afterward gets that careful treatment automatically, without having to remember to add the phrase to each individual question.
>
> The technique-level difference is a system prompt that explicitly separates *how to reason* (system) from *what to solve* (user) — the user prompt stays a plain question either way.

**Task:** solve a multi-step word problem two ways and compare, changing only the system prompt.


### TODO 5.1 — Direct system prompt (final answer only)

Below is a multi-step word problem (feel free to swap in your own, as long as it genuinely requires more than one arithmetic step). Write a **system prompt** that instructs the model to give *only* the final numeric answer, with no explanation -- this is your baseline to compare against.


In [ ]:
word_problem = (
    "A bakery bakes 240 cookies. They sell 3/4 of them in the morning. "
    "In the afternoon, they bake 60 more cookies and sell half of the "
    "total cookies they now have. How many cookies are left at the end "
    "of the day?"
)

# TODO 5.1 -- Write a SYSTEM prompt that instructs the model to answer with
# ONLY the final numeric answer, no explanation. The user prompt is just
# `word_problem`, unchanged.

direct_system_prompt = ...  # TODO: your system prompt string

direct_reply = call_llm(word_problem, system_prompt=direct_system_prompt)
print("Direct answer:", direct_reply)


### TODO 5.2 — Chain-of-thought system prompt

Now write a **new system prompt** that instructs the model to *always* show its reasoning step by step before giving a final answer (e.g. "Whenever you are asked a problem, work through it step by step, showing your reasoning for each step, then state the final answer on its own line at the end."). The user prompt stays exactly `word_problem` — only the system prompt changes.


In [ ]:
# TODO 5.2 -- Write a SYSTEM prompt asking for step-by-step reasoning, THEN
# the final answer. Reuse `word_problem` unchanged as the user prompt.

cot_system_prompt = ...  # TODO: your CoT system prompt string

cot_reply = call_llm(word_problem, system_prompt=cot_system_prompt)
print("Chain-of-thought answer:\n", cot_reply)


### TODO 5.3 — Compare

Work out the correct answer to `word_problem` yourself on paper first. Then, in a comment, note: did `direct_reply` match it? Did `cot_reply` match it? If the two replies disagree, which one shows *where* the reasoning went right or wrong — and why does putting this instruction in the system prompt (rather than the user message) matter once you're sending many different questions through the same assistant?


In [ ]:
# TODO 5.3 -- In a comment:
# 1) What is the correct answer, worked out by hand?
# 2) Did the direct system prompt get it right?
# 3) Did the CoT system prompt get it right?
# 4) Why does putting the reasoning instruction in the system prompt (vs.
#    the user message) matter for an assistant that will field many
#    different user questions?


## 6 · Wrap-up

You've now practiced three core prompting techniques, all driven through the **system prompt**:

- **System prompt engineering** — role, context, constraints, and output format, set once, reshape every answer without touching the user's question.
- **Few-shot prompting** — a handful of labelled examples living in the system prompt teaches a classification pattern that applies to any new user message.
- **Chain-of-thought** — a standing "reason step by step" instruction in the system prompt improves reliability on multi-step problems for every question that follows, not just one.

This is exactly how production assistants are usually built: a carefully engineered system prompt defines the assistant's behavior once, and the user prompt stays simple and changes every turn.
